In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import monai
import torch
import re
import torchvision.transforms as transforms
from glob import glob
from PIL import Image
from monai.transforms import LoadImage

def patients_dicts(folder):
    dicts = []
    patient_paths = glob(os.path.join(folder, "patient*"))

    for patient in patient_paths:
        patient_name = os.path.basename(patient)
        info_paths = glob(os.path.join(patient, "*.cfg"))
        
        with open(info_paths[0], 'r') as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("ED:"):
                        ED = int(line.split(":")[1].strip())
                    elif line.startswith("ES:"):
                        ES = int(line.split(":")[1].strip())
                    elif line.startswith("Group:"):
                        disease = line.split(":")[1].strip()
        
        EDimg_path = glob(os.path.join(patient, f'*{ED}.nii.gz'))[0]
        EDmask_path = glob(os.path.join(patient, f'*{ED}_gt.nii.gz'))[0]
        ESimg_path = glob(os.path.join(patient, f'*{ES}.nii.gz'))[0]
        ESmask_path = glob(os.path.join(patient, f'*{ES}_gt.nii.gz'))[0]

        dicts.append({'ID': patient_name, 'Disease': disease, 'imgED': EDimg_path, 'maskED': EDmask_path, 'imgES': ESimg_path, 'maskES': ESmask_path})
    return dicts

In [5]:
import numpy as np

def par_voxelsize(new_list, dataset):
    for i in range(len(dataset)):
        affED = dataset[i]["imgED"].meta["affine"]
        spED = np.sqrt((affED[:3, :3] ** 2).sum(0))
        new_list.append(spED)
    arr = np.asarray(new_list, dtype=np.float32)
    
    median = arr.median(axis=0)
    standard = arr.std(axis=0)
    maxim = arr.max(axis=0)
    return median, standard, maxim

In [6]:
def par_size(new_list, dataset):
    for i in range(len(dataset)):
        shapeED = dataset[i]["maskED"].shape[1:]
        new_list.append(shapeED)
        
    maxx = 0; maxy = 0; maxz = 0
    for images in range(len(new_list)):
        maxx = max(maxx,new_list[images][0])
        maxy = max(maxy,new_list[images][1])
        maxz = max(maxz,new_list[images][2])
        max_size = [maxx, maxy, maxz]

    return max_size, new_list